# Sheet Music Transposer - HOMR OMR Engine (GPU)

This notebook runs HOMR (Homer's OMR) on Google Colab with free GPU acceleration.
Upload a PDF of sheet music and get MusicXML output.

**Instructions:**
1. Go to Runtime > Change runtime type > Select **T4 GPU**
2. Run all cells in order
3. Upload your PDF when prompted
4. Download the resulting MusicXML files

## 1. Install Dependencies

In [1]:
!pip install -q homr pymupdf music21 verovio

# Verify GPU is available
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.6/292.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 61.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

## 2. Download HOMR Models

In [2]:
# Fix numpy/Python compatibility issue in HOMR's autocrop
import importlib
import homr.autocrop as ac
import inspect

src = inspect.getsource(ac.autocrop)
if 'int(x[1])' in src:
    # Patch the autocrop function
    import homr.autocrop
    original_source = inspect.getsourcefile(homr.autocrop)
    with open(original_source, 'r') as f:
        content = f.read()
    content = content.replace('int(x[1])', 'float(x[1].flat[0])')
    with open(original_source, 'w') as f:
        f.write(content)
    importlib.reload(homr.autocrop)
    print("Patched HOMR autocrop for numpy compatibility")
else:
    print("HOMR autocrop already compatible")

# Download model weights
from homr.main import download_weights
print("Downloading HOMR models (one-time, ~250MB)...")
download_weights()
print("Models ready!")

HOMR autocrop already compatible


ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

## 3. Upload Your PDF

In [ ]:
import os, glob, shutil

# Create working directories
os.makedirs('/content/uploads', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)
os.makedirs('/content/pages', exist_ok=True)

# Find PDF uploaded via sidebar (in /content/)
pdfs = [f for f in glob.glob('/content/*.pdf') if 'sample_data' not in f]
if not pdfs:
    raise FileNotFoundError("No PDF found! Upload a PDF via the Files panel on the left.")

pdf_src = pdfs[0]
pdf_filename = os.path.basename(pdf_src)
pdf_path = f'/content/uploads/{pdf_filename}'
shutil.copy2(pdf_src, pdf_path)

print(f"Using: {pdf_filename} ({os.path.getsize(pdf_path) / 1024:.0f} KB)")

## 4. Convert PDF Pages to Images

In [ ]:
import fitz  # PyMuPDF
from IPython.display import display, Image as IPImage

doc = fitz.open(pdf_path)
num_pages = len(doc)
print(f"PDF has {num_pages} page(s)")

page_images = []
for i in range(num_pages):
    page = doc[i]
    # Render at 300 DPI for good OMR quality
    mat = fitz.Matrix(300/72, 300/72)
    pix = page.get_pixmap(matrix=mat)
    img_path = f'/content/pages/page_{i+1}.png'
    pix.save(img_path)
    page_images.append(img_path)
    print(f"  Page {i+1}: {pix.width}x{pix.height} px")

doc.close()

# Show first page preview
print(f"\nPreview of page 1:")
display(IPImage(filename=page_images[0], width=500))

## 5. Run HOMR on All Pages (GPU Accelerated)

In [ ]:
import time
import shutil
from homr.main import process_image, ProcessingConfig
from homr.xml_generator import XmlGeneratorArguments

config = ProcessingConfig(
    enable_debug=False,
    enable_cache=False,
    write_staff_positions=False,
    read_staff_positions=False,
    selected_staff=-1,  # all staves
)
xml_args = XmlGeneratorArguments(large_page=None, metronome=None, tempo=None)

results = []
total_start = time.time()

for i, img_path in enumerate(page_images):
    print(f"\n{'='*50}")
    print(f"Processing page {i+1}/{num_pages}...")
    page_start = time.time()

    # Copy image to output dir (HOMR writes .musicxml next to input)
    work_dir = f'/content/outputs/page_{i+1}'
    os.makedirs(work_dir, exist_ok=True)
    work_img = f'{work_dir}/page_{i+1}.png'
    shutil.copy2(img_path, work_img)

    try:
        result_staffs = process_image(work_img, config, xml_args)
        elapsed = time.time() - page_start

        musicxml_path = work_img.replace('.png', '.musicxml')
        if os.path.exists(musicxml_path):
            size = os.path.getsize(musicxml_path)
            results.append({
                'page': i+1,
                'status': 'OK',
                'staffs': len(result_staffs),
                'musicxml': musicxml_path,
                'size': size,
                'time': elapsed
            })
            print(f"  OK! {len(result_staffs)} staffs, {size/1024:.0f} KB, {elapsed:.1f}s")
        else:
            results.append({'page': i+1, 'status': 'NO_OUTPUT', 'time': elapsed})
            print(f"  No MusicXML output produced ({elapsed:.1f}s)")

    except Exception as e:
        elapsed = time.time() - page_start
        results.append({'page': i+1, 'status': 'ERROR', 'error': str(e), 'time': elapsed})
        print(f"  ERROR: {e} ({elapsed:.1f}s)")

total_time = time.time() - total_start
success_count = sum(1 for r in results if r['status'] == 'OK')

print(f"\n{'='*50}")
print(f"DONE: {success_count}/{num_pages} pages processed in {total_time:.1f}s")
print(f"\nResults:")
for r in results:
    if r['status'] == 'OK':
        print(f"  Page {r['page']}: {r['staffs']} staffs, {r['size']/1024:.0f} KB ({r['time']:.1f}s)")
    else:
        print(f"  Page {r['page']}: {r['status']} ({r.get('error', '')})")

## 6. Merge Pages into Single MusicXML

In [ ]:
from music21 import converter, stream as m21_stream

successful_pages = [r for r in results if r['status'] == 'OK']

if not successful_pages:
    print("No pages were successfully processed!")
else:
    scores = []
    for r in successful_pages:
        try:
            parsed = converter.parse(r['musicxml'])
            if isinstance(parsed, m21_stream.Score):
                scores.append(parsed)
            elif isinstance(parsed, m21_stream.Opus):
                for s in parsed.scores:
                    scores.append(s)
            else:
                sc = m21_stream.Score()
                sc.append(parsed)
                scores.append(sc)
            print(f"  Page {r['page']}: loaded")
        except Exception as e:
            print(f"  Page {r['page']}: parse error - {e}")

    if len(scores) == 1:
        merged = scores[0]
    elif len(scores) > 1:
        # Merge: append measures from subsequent scores to the first
        merged = scores[0]
        base_parts = list(merged.parts)
        for mvt in scores[1:]:
            mvt_parts = list(mvt.parts)
            for i, bp in enumerate(base_parts):
                if i < len(mvt_parts):
                    for m in mvt_parts[i].getElementsByClass(m21_stream.Measure):
                        bp.append(m)
        print(f"\nMerged {len(scores)} pages")
    else:
        merged = None

    if merged:
        stem = os.path.splitext(pdf_filename)[0]
        output_path = f'/content/outputs/{stem}_homr.musicxml'
        merged.write('musicxml', fp=output_path)
        print(f"\nFinal MusicXML: {output_path}")
        print(f"Size: {os.path.getsize(output_path) / 1024:.0f} KB")

        # Show analysis
        print(f"\nScore Analysis:")
        print(f"  Parts: {len(merged.parts)}")
        for i, part in enumerate(merged.parts):
            print(f"    [{i}] {part.partName or 'Unnamed'}")
            measures = list(part.getElementsByClass(m21_stream.Measure))
            print(f"        Measures: {len(measures)}")

## 7. Transpose (Optional)

In [ ]:
# Configure transposition here
TRANSPOSE = True  # Set to False to skip
TARGET_INSTRUMENT = 'violin'  # violin, viola, cello, flute, clarinet, etc.
TRANSPOSE_INTERVAL = 'P8'    # P8 = octave up, P-8 = octave down, M2 = whole step up
TARGET_CLEF = 'treble'       # treble, bass, alto, tenor

if TRANSPOSE and merged:
    from music21 import clef, instrument, interval

    CLEF_MAP = {
        'treble': clef.TrebleClef,
        'bass': clef.BassClef,
        'alto': clef.AltoClef,
        'tenor': clef.TenorClef,
    }

    INSTRUMENT_MAP = {
        'violin': instrument.Violin,
        'viola': instrument.Viola,
        'cello': instrument.Violoncello,
        'flute': instrument.Flute,
        'clarinet': instrument.Clarinet,
        'piano': instrument.Piano,
    }

    for part in merged.parts:
        # Change clef
        if TARGET_CLEF in CLEF_MAP:
            for old_clef in part.recurse().getElementsByClass(clef.Clef):
                old_clef.activeSite.replace(old_clef, CLEF_MAP[TARGET_CLEF]())
            print(f"Changed clef to {TARGET_CLEF}")

        # Transpose
        if TRANSPOSE_INTERVAL:
            ivl = interval.Interval(TRANSPOSE_INTERVAL)
            part.transpose(ivl, inPlace=True)
            print(f"Transposed by {TRANSPOSE_INTERVAL}")

        # Change instrument
        if TARGET_INSTRUMENT in INSTRUMENT_MAP:
            for old_inst in part.recurse().getElementsByClass(instrument.Instrument):
                old_inst.activeSite.replace(old_inst, INSTRUMENT_MAP[TARGET_INSTRUMENT]())
            print(f"Changed instrument to {TARGET_INSTRUMENT}")

    stem = os.path.splitext(pdf_filename)[0]
    transposed_path = f'/content/outputs/{stem}_{TARGET_INSTRUMENT}_transposed.musicxml'
    merged.write('musicxml', fp=transposed_path)
    print(f"\nTransposed output: {transposed_path}")
    print(f"Size: {os.path.getsize(transposed_path) / 1024:.0f} KB")
else:
    print("Skipping transposition")

## 8. Render to PDF (Optional)

In [ ]:
import verovio

# Pick which file to render
if TRANSPOSE and 'transposed_path' in dir():
    render_source = transposed_path
else:
    render_source = output_path

with open(render_source, 'r') as f:
    musicxml_str = f.read()

vrvToolkit = verovio.toolkit()
vrvToolkit.loadData(musicxml_str)

page_count = vrvToolkit.getPageCount()
print(f"Rendering {page_count} page(s) as SVG...")

from IPython.display import display, HTML, SVG

for i in range(1, min(page_count + 1, 6)):  # Show up to 5 pages
    svg = vrvToolkit.renderToSVG(i)
    display(HTML(f"<h3>Page {i}</h3>"))
    display(SVG(data=svg))

if page_count > 5:
    print(f"... and {page_count - 5} more pages (download MusicXML for full score)")

## 9. Download Results

In [ ]:
from google.colab import files
import glob

print("Available output files:")
output_files = glob.glob('/content/outputs/*.musicxml')
for f in output_files:
    size = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f)} ({size:.0f} KB)")

print("\nDownloading all output files...")
for f in output_files:
    files.download(f)

# Also download per-page results
page_files = glob.glob('/content/outputs/page_*/*.musicxml')
if page_files:
    print(f"\nPer-page MusicXML files ({len(page_files)} files):")
    for f in page_files:
        print(f"  {f}")
    download_pages = input("Download individual page files too? (y/n): ")
    if download_pages.lower() == 'y':
        for f in page_files:
            files.download(f)

## 10. Compare: Run Audiveris Too (Optional)

Run this cell to also try Audiveris and compare results.

In [ ]:
COMPARE_AUDIVERIS = False  # Set to True to also run Audiveris

if COMPARE_AUDIVERIS:
    print("Installing Audiveris...")
    !apt-get install -y -q default-jre > /dev/null 2>&1
    !pip install -q audiveris 2>/dev/null || echo "Audiveris pip not available, using Docker"

    # Check if Java is available
    !java -version 2>&1 | head -1

    print("\nNote: Audiveris on Colab requires manual setup.")
    print("HOMR is the recommended OMR engine for Colab.")
else:
    print("Skipping Audiveris comparison. Set COMPARE_AUDIVERIS = True to enable.")